# Seguridad del entrenamiento federado

Endurecemos el esquema de aprendizaje federado del proyecto (el mismo clasificador
softmax entrenado desde cero en numpy, sobre el dataset `digits`, de
`notebooks/06_federated_learning.ipynb`).

**Riesgo:** FedAvg promedia los updates de todos los clientes por igual, confiando en que
ninguno estÃ¡ comprometido. Un Ãºnico **cliente malicioso** que manda un update escalado e
invertido (*model poisoning*) puede arruinar el modelo global entero, porque el promedio
simple no tiene forma de distinguir una actualizaciÃ³n legÃ­tima de una maliciosa.

**Controles que agregamos:** *clipping* de la norma del update de cada cliente (acota
cuÃ¡nto puede influir un solo cliente), agregaciÃ³n robusta por mediana por coordenada (en
vez de promedio simple), y la combinamos con la privacidad diferencial que ya tenÃ­amos.

In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rng = np.random.RandomState(42)
X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
N_FEAT, N_CLS = X.shape[1], 10


def init_modelo():
    return np.zeros((N_FEAT, N_CLS)), np.zeros(N_CLS)


def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)


def onehot(y):
    O = np.zeros((len(y), N_CLS))
    O[np.arange(len(y)), y] = 1
    return O


def paso_sgd(W, b, X, y, lr):
    P = softmax(X @ W + b)
    D = (P - onehot(y)) / len(y)
    return W - lr * (X.T @ D), b - lr * D.sum(axis=0)


def accuracy(W, b, X=Xte, y=yte):
    return (np.argmax(X @ W + b, axis=1) == y).mean()


def particionar_iid(K):
    idx = rng.permutation(len(Xtr))
    return [(Xtr[c], ytr[c]) for c in np.array_split(idx, K)]


K_CLIENTES = 5
clientes = particionar_iid(K_CLIENTES)
print("Clientes:", [len(Xc) for Xc, _ in clientes])

Clientes: [270, 270, 269, 269, 269]


## 1. Un cliente malicioso arruina FedAvg con promedio simple

El cliente 0 entrena localmente como cualquier otro, pero antes de mandar su update lo
**escala e invierte** â€” simulando que un participante de la red federada fue comprometido,
o es directamente un atacante que se uniÃ³ al proceso.

In [2]:
def entrenar_local(W_glob, b_glob, Xc, yc, epocas=25, lr=0.5):
    W, b = W_glob.copy(), b_glob.copy()
    for _ in range(epocas):
        W, b = paso_sgd(W, b, Xc, yc, lr)
    return W, b, len(Xc)


def entrenar_local_malicioso(W_glob, b_glob, Xc, yc, epocas=25, lr=0.5, factor_ataque=-10.0):
    """Entrena normal, pero escala e invierte el update antes de mandarlo
    (model poisoning: el objetivo es arrastrar el modelo global en la direcciÃ³n opuesta)."""
    W, b, n = entrenar_local(W_glob, b_glob, Xc, yc, epocas, lr)
    dW, db = (W - W_glob) * factor_ataque, (b - b_glob) * factor_ataque
    return W_glob + dW, b_glob + db, n


def fedavg_simple(actualizaciones):
    n = sum(nk for _, _, nk in actualizaciones)
    W = sum((nk / n) * Wk for Wk, _, nk in actualizaciones)
    b = sum((nk / n) * bk for _, bk, nk in actualizaciones)
    return W, b


def federado(clientes, agregador, R=20, epocas=25, lr=0.5, cliente_malicioso=None):
    W, b = init_modelo()
    historia = []
    for _ in range(R):
        ups = []
        for i, (Xc, yc) in enumerate(clientes):
            if i == cliente_malicioso:
                ups.append(entrenar_local_malicioso(W, b, Xc, yc, epocas, lr))
            else:
                ups.append(entrenar_local(W, b, Xc, yc, epocas, lr))
        W, b = agregador(ups)
        historia.append(accuracy(W, b))
    return W, b, historia


_, _, hist_sin_ataque = federado(clientes, fedavg_simple)
_, _, hist_con_ataque = federado(clientes, fedavg_simple, cliente_malicioso=0)

print(f"FedAvg sin ataque, accuracy final:                {hist_sin_ataque[-1]:.3f}")
print(f"FedAvg con 1 cliente malicioso (promedio simple):  {hist_con_ataque[-1]:.3f}  <- colapsa")

FedAvg sin ataque, accuracy final:                0.971
FedAvg con 1 cliente malicioso (promedio simple):  0.000  <- colapsa


## 2. AgregaciÃ³n robusta: mediana por coordenada

En vez de promediar, tomamos la **mediana** de cada coordenada de los pesos entre todos
los clientes. Un Ãºnico cliente malicioso ya no puede arrastrar el agregado, porque la
mediana ignora valores extremos aislados â€” necesitarÃ­a comprometer a mÃ¡s de la mitad de
los clientes para mover el resultado.

In [3]:
def fedavg_mediana(actualizaciones):
    Ws = np.stack([Wk for Wk, _, _ in actualizaciones])
    bs = np.stack([bk for _, bk, _ in actualizaciones])
    return np.median(Ws, axis=0), np.median(bs, axis=0)


_, _, hist_robusto = federado(clientes, fedavg_mediana, cliente_malicioso=0)
print(f"Mediana por coordenada, con 1 cliente malicioso: {hist_robusto[-1]:.3f}  <- resiste")

Mediana por coordenada, con 1 cliente malicioso: 0.931  <- resiste


## 3. Clipping de norma + privacidad diferencial, combinados con la agregaciÃ³n robusta

Sumamos dos defensas mÃ¡s, independientes entre sÃ­: **clipping** acota la norma del update
de cada cliente antes de agregar (si un cliente manda un update de magnitud anormal,
aunque no sea exactamente el ataque simulado arriba, se recorta); **ruido gaussiano**
agregado al update evita que un observador externo infiera datos del cliente a partir de
su actualizaciÃ³n (privacidad diferencial, igual mecanismo que ya usamos en
`notebooks/06_federated_learning.ipynb`, ahora combinado con la agregaciÃ³n robusta).

In [4]:
def entrenar_local_protegido(W_glob, b_glob, Xc, yc, epocas, lr, clip_norm, noise_std):
    W, b, n = entrenar_local(W_glob, b_glob, Xc, yc, epocas, lr)
    dW, db = W - W_glob, b - b_glob

    norma = np.sqrt((dW ** 2).sum() + (db ** 2).sum())
    factor = min(1.0, clip_norm / (norma + 1e-12))
    dW, db = dW * factor, db * factor

    if noise_std > 0:
        dW = dW + rng.normal(0, noise_std, dW.shape)
        db = db + rng.normal(0, noise_std, db.shape)

    return W_glob + dW, b_glob + db, n


def entrenar_local_malicioso_protegido(W_glob, b_glob, Xc, yc, epocas, lr, clip_norm, noise_std,
                                        factor_ataque=-10.0):
    W, b, n = entrenar_local(W_glob, b_glob, Xc, yc, epocas, lr)
    dW, db = (W - W_glob) * factor_ataque, (b - b_glob) * factor_ataque
    # el clipping se aplica IGUAL a todos, incluido el cliente malicioso
    norma = np.sqrt((dW ** 2).sum() + (db ** 2).sum())
    factor = min(1.0, clip_norm / (norma + 1e-12))
    dW, db = dW * factor, db * factor
    return W_glob + dW, b_glob + db, n


def federado_protegido(clientes, agregador, R, epocas, lr, clip_norm, noise_std, cliente_malicioso=None):
    W, b = init_modelo()
    historia = []
    for _ in range(R):
        ups = []
        for i, (Xc, yc) in enumerate(clientes):
            if i == cliente_malicioso:
                ups.append(entrenar_local_malicioso_protegido(
                    W, b, Xc, yc, epocas, lr, clip_norm, noise_std
                ))
            else:
                ups.append(entrenar_local_protegido(W, b, Xc, yc, epocas, lr, clip_norm, noise_std))
        W, b = agregador(ups)
        historia.append(accuracy(W, b))
    return W, b, historia


_, _, hist_completo = federado_protegido(
    clientes, fedavg_mediana, R=20, epocas=25, lr=0.5,
    clip_norm=1.0, noise_std=0.01, cliente_malicioso=0,
)
print(
    "Mediana + clipping + DP (noise_std=0.01), con 1 cliente malicioso: "
    f"{hist_completo[-1]:.3f}"
)
print(f"\nResumen:")
print(f"  sin ataque, FedAvg simple:                 {hist_sin_ataque[-1]:.3f}")
print(f"  con ataque, FedAvg simple:                 {hist_con_ataque[-1]:.3f}  (colapsa)")
print(f"  con ataque, mediana sola:                  {hist_robusto[-1]:.3f}  (resiste)")
print(f"  con ataque, mediana + clipping + DP:        {hist_completo[-1]:.3f}  (resiste y protege privacidad)")

Mediana + clipping + DP (noise_std=0.01), con 1 cliente malicioso: 0.936

Resumen:
  sin ataque, FedAvg simple:                 0.971
  con ataque, FedAvg simple:                 0.000  (colapsa)
  con ataque, mediana sola:                  0.931  (resiste)
  con ataque, mediana + clipping + DP:        0.936  (resiste y protege privacidad)


## 4. ReflexiÃ³n

**Â¿Por quÃ© el promedio simple es tan frÃ¡gil ante un solo cliente malicioso?**
FedAvg pondera cada update por la cantidad de datos del cliente, pero no evalÃºa si ese
update es *razonable* en relaciÃ³n a los demÃ¡s. Un cliente que manda un update de magnitud
10 veces mayor y en direcciÃ³n opuesta arrastra el promedio hacia esa direcciÃ³n sin
oposiciÃ³n â€” con 5 clientes, uno malicioso ya representa el 20% del peso agregado antes de
ponderar, suficiente para dominar el resultado si su update es extremo.

**Â¿QuÃ© protege el clipping y quÃ© protege la privacidad diferencial (son cosas distintas)?**
El clipping protege contra la **magnitud** de un update: acota cuÃ¡nto puede empujar el
modelo global cualquier cliente individual, sea por un ataque deliberado o un bug local.
La privacidad diferencial protege algo completamente distinto â€” la **informaciÃ³n** que un
update revela sobre los datos privados del cliente que lo generÃ³. Un cliente podrÃ­a mandar
un update de magnitud perfectamente razonable (pasa el clipping sin problema) que igual
filtre, por su forma exacta, detalles reconstruibles sobre sus datos de entrenamiento; el
ruido gaussiano hace que esa reconstrucciÃ³n sea estadÃ­sticamente mÃ¡s difÃ­cil, al costo de
una seÃ±al ligeramente mÃ¡s ruidosa para el modelo global.

**Â¿QuÃ© elemento de SAIF cubre este control?**
La agregaciÃ³n robusta (mediana en vez de promedio) es la aplicaciÃ³n directa de *detectar y
actuar*: en vez de asumir que todos los participantes son de confianza, el sistema tolera
que algunos no lo sean sin que eso arruine el resultado agregado â€” es la misma lÃ³gica de
"defensa en profundidad" que en REST se traduce en rate limiting, pero aplicada al
problema especÃ­fico de agregar contribuciones de fuentes no completamente confiables.